# 🇹🇲 Туркменский морфологический сегментатор v4

**Ключевые возможности:**
- 🗂️ **Выбор файла через виджет** — не нужно редактировать код
- 🚀 **Кнопка «Запустить всё»** — одно нажатие запускает ячейки 5→9
- 📊 Суффиксный индекс (форма→сегментация): `dyrdy→dyr@@dy`
- 🔤 Нормализация ň→ñ, нормализатор имён собственных
- ♻️ Ячейка 9 автоматически перезагружает словари

**Файлы в папке `SEGMENTATION_TURKM/`:**
- `turkm_stems_final.xlsx`, `turkm_endings_final.xlsx`, `stopwords_turkm.txt`
- любые `.txt` файлы для сегментации

**Порядок работы:**
1. Запустить ячейку 1 (Drive)
2. Запустить ячейку 2 (установка)
3. Запустить ячейку 4 → выбрать файл → нажать **🚀 Запустить сегментацию**
> Ячейки 5–9 запустятся автоматически!


## 📁 Ячейка 1 — Монтирование Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
curr_dir = "/content/drive/MyDrive/SEGMENTATION_TURKM"
%cd "$curr_dir"
print(f"✅ Рабочая директория: {curr_dir}")

Mounted at /content/drive
/content/drive/MyDrive/SEGMENTATION_TURKM
✅ Рабочая директория: /content/drive/MyDrive/SEGMENTATION_TURKM


## 📦 Ячейка 2 — Установка зависимостей

In [ ]:
!pip install openpyxl sacrebleu tqdm --quiet
print("✅ Зависимости установлены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.8 MB/s eta 0:00:00
✅ Зависимости установлены


## ⚙️ Ячейка 3 — Конфигурация путей к файлам

In [ ]:
# ⚠️  ВНИМАНИЕ: конфигурация перенесена в начало Ячейки 4.
# Эту ячейку запускать НЕ ОБЯЗАТЕЛЬНО — настройки уже встроены.
# Если хотите переопределить пути — раскомментируйте и измените:

# STEMS_FILE      = "turkm_stems_final.xlsx"
# ENDINGS_FILE    = "turkm_endings_final.xlsx"
# STOPWORDS_FILE  = "stopwords_turkm.txt"
# INPUT_FILE      = "turkm_200_only.txt"
# OUTPUT_DIR      = "segmented_output"
# CHUNK_SIZE      = 5000
# COL_ENDINGS_FORM = 0
# COL_ENDINGS_SEG  = 1

print('ℹ️  Конфигурация встроена в Ячейку 4 — эту ячейку пропустите.')


ℹ️  Конфигурация встроена в Ячейку 4 — эту ячейку пропустите.


## 📚 Ячейка 4 — Загрузка словарей + выбор входного файла

In [ ]:
# =============================================================
# КОНФИГУРАЦИЯ — выберите входной файл из списка
# =============================================================
import os, glob, ipywidgets as widgets, re, time, openpyxl
from tqdm import tqdm
from IPython.display import display

STEMS_FILE      = "turkm_stems_final.xlsx"
ENDINGS_FILE    = "turkm_endings_final.xlsx"
STOPWORDS_FILE  = "stopwords_turkm.txt"
CHUNK_SIZE      = 5000
COL_ENDINGS_FORM = 0
COL_ENDINGS_SEG  = 1

# Автоматически находим все .txt файлы в текущей директории
txt_files = sorted(glob.glob('*.txt'))
if not txt_files:
    print('⚠️  Нет .txt файлов в текущей папке!')
else:
    w_file = widgets.Dropdown(
        options=txt_files,
        description='Входной файл:',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='450px')
    )
    w_out = widgets.Text(
        value='',
        description='Папка вывода:',
        placeholder='(оставьте пустым — автоматически)',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='450px')
    )
    w_btn = widgets.Button(
        description='✅ Применить',
        button_style='success',
        layout=widgets.Layout(width='150px')
    )
    w_label = widgets.Label(value='')

    def on_apply(b):
        global INPUT_FILE, OUTPUT_DIR
        INPUT_FILE = w_file.value
        OUTPUT_DIR = w_out.value.strip() or (os.path.splitext(INPUT_FILE)[0] + '_segmented')
        w_label.value = f'✅ INPUT_FILE = "{INPUT_FILE}"  |  OUTPUT_DIR = "{OUTPUT_DIR}"'

    w_btn.on_click(on_apply)

    # Установить дефолтные значения сразу
    INPUT_FILE = txt_files[0]
    OUTPUT_DIR = os.path.splitext(INPUT_FILE)[0] + '_segmented'

    display(widgets.VBox([
        widgets.HTML('<b>🗂️ Выберите файл для сегментации:</b>'),
        w_file, w_out, w_btn, w_label
    ]))
    print(f'📄 По умолчанию выбран: "{INPUT_FILE}"')
    print(f'📁 Папка вывода:        "{OUTPUT_DIR}"')
    print('   (нажмите ✅ Применить чтобы изменить)')

import re
import os
import time
import openpyxl
from tqdm import tqdm

# ── НОРМАЛИЗАЦИЯ НОСОВОГО СИМВОЛА (КРИТИЧНО!) ─────────────────────────────
# В текстах: ň (U+0148, LATIN SMALL LETTER N WITH CARON)
# В таблице: ñ (U+00F1, LATIN SMALL LETTER N WITH TILDE)
# Нормализуем ň→ñ перед любым поиском по таблице.
def norm(s):
    """ň (U+0148) → ñ (U+00F1) для совпадения с таблицей окончаний."""
    return str(s).replace('\u0148', '\u00f1').replace('\u0147', '\u00d1')


# ── 4.1 ЗАГРУЗКА ОКОНЧАНИЙ + СУФФИКСНЫЙ ИНДЕКС ───────────────────────────
def load_endings_with_suffix_index(file_name):
    wb = openpyxl.load_workbook(file_name)
    sh = wb.active
    endings_form = []
    endings_seg  = []
    for row in sh.iter_rows(min_row=2, values_only=True):
        form = row[COL_ENDINGS_FORM]
        seg  = row[COL_ENDINGS_SEG]
        if form is None or seg is None:
            continue
        form = norm(str(form).replace('\ufeff', '').strip())
        seg  = norm(str(seg).replace('\ufeff', '').strip())
        endings_form.append(form)
        endings_seg.append(seg)

    # Суффиксный индекс: все суффиксы сегментации → сегментированное окончание
    suffix_index = {}
    seen_seg = set()
    for form, seg in zip(endings_form, endings_seg):
        # Маппинг форма (без @@) → сегментация (с @@)
        # Это ключевое: 'dyrdy' → 'dyr@@ dy', 'larda' → 'lar@@ da' и т.д.
        if form and form not in suffix_index:
            suffix_index[form] = seg
        if seg in seen_seg:
            continue
        seen_seg.add(seg)
        parts = seg.split('@@ ')
        for start in range(len(parts)):
            key = '@@ '.join(parts[start:])
            if key not in suffix_index:
                suffix_index[key] = '@@ '.join(parts[start:])

    print(f"  ✅ Загружено окончаний: {len(endings_form)} форм")
    print(f"  ✅ Суффиксных ключей:   {len(suffix_index)}")
    return endings_form, endings_seg, suffix_index


# ── 4.2 ЗАГРУЗКА СТЕМОВ ──────────────────────────────────────────────────
def load_stems(file_name):
    wb = openpyxl.load_workbook(file_name)
    sh = wb.active
    stems_set = set()
    for row in sh.iter_rows(min_row=1, values_only=True):
        val = row[0]
        if val is None:
            continue
        stem = norm(str(val).replace('\ufeff', '').strip())
        if stem:
            stems_set.add(stem.lower())
    print(f"  ✅ Загружено стемов: {len(stems_set)}")
    return stems_set


# ── 4.3 ЗАГРУЗКА СТОП-СЛОВ ───────────────────────────────────────────────
def load_stopwords(file_name):
    with open(file_name, 'r', encoding='utf-8') as f:
        stop_words = set(norm(line.strip()) for line in f if line.strip())
    print(f"  ✅ Загружено стоп-слов: {len(stop_words)}")
    return stop_words



# ── 4.4 ЗАГРУЗКА PROPER_NORM ИЗ ФАЙЛА ───────────────────────────────────
def load_proper_norm(file_name='proper_norm.json'):
    """Загружает словарь имён собственных из JSON-файла на Drive.
    Это позволяет обновлять имена без изменения ноутбука."""
    import os
    if not os.path.exists(file_name):
        print(f'  ⚠️  {file_name} не найден, используется встроенный PROPER_NORM')
        return None
    with open(file_name, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'  ✅ Загружен proper_norm.json: {len(data)} записей')
    return data

# ── ЗАГРУЖАЕМ ВСЁ ────────────────────────────────────────────────────────
print("📚 Загрузка словарей...")
endings_form, endings_seg, SUFFIX_INDEX = load_endings_with_suffix_index(ENDINGS_FILE)
STEMS_SET   = load_stems(STEMS_FILE)
STOP_WORDS  = load_stopwords(STOPWORDS_FILE)
print("\n✅ Все словари загружены")


# =============================================================
# 🚀 КНОПКА "ЗАПУСТИТЬ ВСЁ" — нажмите после выбора файла
# =============================================================
from IPython.display import display, Javascript
import ipywidgets as widgets

def run_all_cells(b):
    """Запускает ячейки 5→6→7→8→9 последовательно."""
    w_run_label.value = '⏳ Запускаю ячейки 5→6→7→8→9...'
    display(Javascript('''
        // Находим все ячейки кода и запускаем начиная с 5-й (индекс 4 среди code-ячеек)
        var cells = Jupyter.notebook.get_cells();
        var codeIdx = 0;
        var toRun = [];
        for (var i = 0; i < cells.length; i++) {
            if (cells[i].cell_type === 'code') {
                codeIdx++;
                // Запускаем ячейки 5, 6, 7, 8, 9 (code-ячейки №4..8 считая с 0)
                if (codeIdx >= 4 && codeIdx <= 8) {
                    toRun.push(i);
                }
            }
        }
        // Запускаем последовательно
        function runNext(idx) {
            if (idx >= toRun.length) {
                console.log('✅ Все ячейки выполнены');
                return;
            }
            Jupyter.notebook.execute_cell(toRun[idx]);
            // Ждём завершения и запускаем следующую
            var cell = cells[toRun[idx]];
            var checkDone = setInterval(function() {
                if (!cell.running) {
                    clearInterval(checkDone);
                    runNext(idx + 1);
                }
            }, 500);
        }
        runNext(0);
    '''))
    w_run_label.value = '✅ Запуск начат! Следите за выводом ячеек ниже.'

w_run_btn = widgets.Button(
    description='🚀 Запустить сегментацию (5→6→7→8→9)',
    button_style='primary',
    layout=widgets.Layout(width='350px', height='40px')
)
w_run_label = widgets.Label(value='')
w_run_btn.on_click(run_all_cells)

display(widgets.VBox([
    widgets.HTML('<hr><b>После выбора файла нажмите кнопку:</b>'),
    w_run_btn,
    w_run_label
]))


📄 По умолчанию выбран: "chunk_turkm.txt"
📁 Папка вывода:        "chunk_turkm_segmented"
   (нажмите ✅ Применить чтобы изменить)
📚 Загрузка словарей...
  ✅ Загружено окончаний: 2416 форм
  ✅ Суффиксных ключей:   2053
  ✅ Загружено стемов: 37620
  ✅ Загружено стоп-слов: 84

✅ Все словари загружены


## 🔤 Ячейка 5 — Нормализатор имён собственных (Пункт 4, кат. Б)

In [ ]:
# =============================================================
# ПУНКТ 4 — НОРМАЛИЗАТОР ИМЁН СОБСТВЕННЫХ
# Решает проблему имён с падежными окончаниями:
#   Kazahyň        → стем Kazah  + yň
#   Russiýanyň     → стем Russiýa + nyň
#   Kazahystanda   → стем Kazahystan + da
# =============================================================

# Загружаем PROPER_NORM из JSON файла (обновляется без изменения ноутбука)
import os, json as _json
_pn_file = 'proper_norm.json'
if os.path.exists(_pn_file):
    with open(_pn_file, 'r', encoding='utf-8') as _f:
        PROPER_NORM = _json.load(_f)
    print(f'✅ PROPER_NORM загружен из {_pn_file}: {len(PROPER_NORM)} записей')
else:
    print(f'⚠️  {_pn_file} не найден — используется встроенный словарь')
    # Встроенный словарь (резервный)
    PROPER_NORM = {

    # Страны и регионы
    'Kazahystan':   'Kazahystan',
    'Kazahyst':     'Kazahystan',   # обрезанная форма
    'Kazahysta':    'Kazahystan',   # обрезанная форма
    'Kazah':        'Kazahystan',
    'Russiýa':      'Russiýa',
    'Müsür':        'Müsür',
    'Awganystan':   'Awganystan',
    'Britaniýa':    'Britaniýa',
    'Hindistan':    'Hindistan',
    'Ýaponiýa':     'Ýaponiýa',
    'Özbegistan':   'Özbegistan',
    'Gürcistan':    'Gürcistan',
    'Aziýa':        'Aziýa',        # Азия
    'Gündogar':     'Gündogar',     # Восток (как топоним: Orta Gündogar)
    'Orta':         'Orta',         # Orta Aziýa / Orta Gündogar
    # Города
    'Moskwa':       'Moskwa',
    'Taraz':        'Taraz',
    'Akmola':       'Akmola',
    'Akto':         'Aktobe',       # обрезанная форма Aktobe
    # Праздники и события
    'Pesah':        'Pesah',        # Пасха
    # Люди (фамилии)
    'Nazarbaýew':   'Nazarbaýew',
    'Karimow':      'Karimow',
    'Burkhalter':   'Burkhalter',
    'Saudabaýew':   'Saudabaýew',
    'Doskalyyew':   'Doskalyyew',
    'Sagandykow':   'Sagandykow',
    'Mubarag':      'Mubarag',
    'Singh':        'Singh',
    'Ellouş':       'Ellouş',
    'Rýjkow':       'Rýjkow',
    # Организации и аббревиатуры
    'ABŞ':          'ABŞ',
    'NATO':         'NATO',
    'Kongres':      'Kongres',
    'Patriarh':     'Patriarh',
    'Kadyrov':   'Kadyrov',
    'Sats':   'Sats',
    'Krymskaýa':   'Krymskaýa',
    'Lukoil':   'Lukoil',
    'Alekperov':   'Alekperov',
    'Terroristler':   'Terroristler',
    'Müsüre':   'Müsür',
    # Имена
    'Ýokar':   'Ýokary',
    'Häzirk':   'Häzirki',
    'Soňk':   'Soňky',
    'Kajamzharov':   'Kajamzharov',
    'Kojamýarov':   'Kojamýarov',
    'Françesko':   'Françesko',
    'Franjiall':   'Franjialli',
    'Franjialli':   'Franjialli',
    'Kadyrov':   'Kadyrov',
    'Sats':   'Sats',
    'Krymskaýa':   'Krymskaýa',
    'Lukoil':   'Lukoil',
    'Alekperov':   'Alekperov',
    'Terroristler':   'Terroristler',
    'Müsüre':   'Müsür',
    'Kazymzharov':   'Kazymzharov',
    'Kors':   'Kors',
    'Qaljat':   'Qaljat',
    # Имена
    'Nýu':   'Nýu',
    'Orleans':   'Orleans',
    'Amerika':   'Amerika',
    'Afrika':   'Afrika',
    'Arab':   'Arab',
    'Muhammed':   'Muhammed',
    'Kaşagan':   'Kaşagan',
    'Qaraçaganak':   'Qaraçaganak',
    'Kazakstan':   'Kazahystan',
    'Şeyh':   'Şeyh',
    'Rawil':   'Rawil',
    'Gaýnutdin':   'Gaýnutdin',
    'Görüşmeler':   'Görüşmeler',
    'Gürcista':   'Gürcistan',
    'Komissarlar':   'Komissarlar',
    'Hindista':   'Hindistan',
    'Hos':   'Hosni',
    'Şon':   'Şonuň',
    'Ýaradyj':   'Ýaradyjy',
    'Arystawan':   'Arystawan',
    'Bahmutova':   'Bahmutova',
    'Janburşyn':   'Janburşyn',
    'Sagyngal':   'Sagyngal',
    'Sultonov':   'Sultonov',
    'Beysenbaýew':   'Beysenbaýew',
    'Jakypov':   'Jakypov',
    'Karin':   'Karin',
    'Gyrat':   'Gyrat',
    'Zybergaliýew':   'Zybergaliýew',
    'Gyrgyzysta':   'Gyrgyzystan',
    'Nur':   'Nur',
    'Orhon':   'Orhon',
    'Ýeniseý':   'Ýeniseý',
    'Bütindünýä':   'Bütindünýä',
    'İslam':   'İslam',
    'Abdel':   'Abdel',
    'Muhsen':   'Muhsen',
    'Otan':   'Otan',
    'Ben':   'Ben',
    'Eý':   'Eý',
    'Nursultan':    'Nursultan',
    'Manmohan':     'Manmohan',
    'Pierre':       'Pierre',
    'Nikolay':      'Nikolay',
    'Didier':       'Didier',
    'Nazarbaýeba':   'Nazarbaýew',
    'Nazarbaýewa':   'Nazarbaýew',
    'Jambol':   'Jambol',
    'Kanu':   'Kanunyň',
    'Parlamentaral':   'Parlamentaraly',
    'Ýaradyja':   'Ýaradyjy',
    'Ýewropa':   'Ýewropa',
    'Federasiýa':   'Federasiýa',
    'Şurasy':   'Şura',
    'Rýjkov':   'Rýjkov',
    'Boldykow':   'Boldykow',
    'Kazımdarov':   'Kazımdarov',
    'Awganist':   'Awganystan',
    'Eleusyn':   'Eleusyn',
    'Doskalyyev':   'Doskalyyev',
    'Çen':   'Çeni',
    'Karimov':   'Karimov',
    'Asambleyas':   'Assambleýa',
    'Konstitutsion':   'Konstitutsion',
    'Ýüze':   'Ýüze',
    'Nazarbaev':   'Nazarbaýew',
    'Kazaxstan':   'Kazahystan',
    'Almat':   'Almaty',
    'Astana':   'Astana',
    'BMG':   'BMG',
    'Oblastyn':   'Oblast',
    'Kızılor':   'Kızılorda',
    'Nurl':   'Nur-Sultan',
    'Majlisa':   'Majlis',
    'EKSPO':   'EXPO',
    'Abdykalikova':   'Abdykalikova',
    'Karag':   'Karaganda',
    'Zymbal':   'Zymbal',
    'ÝEAK':   'ÝEAK',
    'Ýuwaş':   'Ýuwaş',
    'Kazaýst':   'Kazahystan',
    'Ýaponiýadak':   'Ýaponiýa',
    'Serbiýa':   'Serbiýa',
    'Ýewrosiýa':   'Ýewrosiýa',
    'Damu':   'Damu',
    'Nazarbaýa':   'Nazarbaýew',
    'Massymov':   'Massymov',
    'Sagintayev':   'Sagintayev',
    'Pawlodar':   'Pawlodar',
    'XXI':   'XXI',
    'Masymov':   'Masymov',
    'Sultanov':   'Sultanov',
    'Pavlodar':   'Pavlodar',
    'KADEX':   'KADEX',
    'Majelis':   'Majelis',
    'Mangistau':   'Mangistau',
    'MÝM':   'MÝM',
    'Baýterek':   'Baýterek',
    'Mejilis':   'Mejilis',
    'Birlig':   'Birlik',
    'Kaledoniýa':   'Kaledoniýa',
    'Masimov':   'Masimov',
    'Guramasy':   'Gurama',
    'Atyra':   'Atyrau',
    'Mangistaw':   'Mangistaw',
    'ÝEAEA':   'ÝEAEA',
    'Gürgizistan':   'Gyrgyzystan',
    'Merýel':   'Merýel',
    'Isesekeshew':   'Isesekeshew',
    'GDA':   'GDA',
    'MKS':   'MKS',
    'Kazachsta':   'Kazahystan',
    'Trend':   'Trend',
    'Tokayew':   'Tokayew',
    'Jomar':   'Jomar',
    'Mamin':   'Mamin',
    'Kassym':   'Kassym',
    'Şymkent':   'Şymkent',
    'Kostana':   'Kostanay',
    'Atamkulov':   'Atamkulov',
    'Türkistan':   'Türkistan',
    'Ahmetov':   'Ahmetov',
    'Aktau':   'Aktau',
    'Vasilenko':   'Vasilenko',
    'Abdrahmanov':   'Abdrahmanov',
    'Kazakhst':   'Kazahystan',
    'Atameken':   'Atameken',
    'Kairat':   'Kairat',
    'Dosaýew':   'Dosaýew',
    'Kazachstan':   'Kazahystan',
    'Özbekista':   'Özbekistan',
    'Kökşetaw':   'Kökşetaw',
    'Başlyg':   'Başlyk',
    'Askar':   'Askar',
    'Abdirahmanow':   'Abdirahmanow',
    'Şarqi':   'Şarqi',
    'Jakupov':   'Jakupov',
    'Tajikistan':   'Tajikistan',
    'Dilenov':   'Dilenov',
    'Elçi':   'Elçi',
    'EKM':   'EKM',
    'Afganysta':   'Awganystan',
    'Kostaý':   'Kostaý',
    'BTM':   'BTM',
    'Ural':   'Ural',
    'Türkmenista':   'Türkmenistan',
    'Ustalyk':   'Ustalyk',
    'Nazarbaýewi':   'Nazarbaýew',
    'Şengal':   'Şengal',
    'Ştat':   'Ştat',
    'Akor':   'Akor',
    'NEP':   'NEP',
    'Energetik':   'Energetik',
    'Saparbayew':   'Saparbayew',
    'MUK':   'MUK',
    'Süleymenov':   'Süleymenov',
    'Baybek':   'Baybek',
    'Gürgizstan':   'Gyrgyzystan',
    'MEA':   'MEA',
    'Ýehowa':   'Ýehowa',
    'Slovakiýa':   'Slovakiýa',
    'Atyraw':   'Atyrau',
    'Ruslan':   'Ruslan',
    'Esimov':   'Esimov',
    'Süýed':   'Süýed',
    'Sawd':   'Sawd',
    'Mançestaw':   'Mançestaw',
    'Niderlandiýa':   'Niderlandiýa',
    'Modernizatsiya':   'Modernizatsiya',
    'Türkiýä':   'Türkiýe',
    'KazAgro':   'KazAgro',
    'Kazakhstan':   'Kazahystan',
    'Kaspiý':   'Kaspiý',
    'Kazahst':   'Kazahystan',
    'KTJ':   'KTJ',
    'Ýork':   'Nýu-Ýork',
    'GKS':   'GKS',
    'Kelymbetov':   'Kelymbetov',
    'Smailov':   'Smailov',
    'Ştatlar':   'Ştatlar',
    'Business':   'Business',
    'Atyraý':   'Atyrau',
    'Smaylov':   'Smaylov',
    'MHD':   'MHD',
    'Sankt':   'Sankt',
    'Kazaşst':   'Kazahystan',
    'Respublikas':   'Respublika',
    'Belarusiýa':   'Belarusiýa',
    'Kaspiýa':   'Kaspiý',
    'Nazarebaev':   'Nazarbaýew',
    'Mejlisi':   'Majlis',
    'Ewropa':   'Ýewropa',
    'Kapparov':   'Kapparov',
    'Bolaşaq':   'Bolaşaq',
    'Dab':   'Dab',
    }

# Туркменские падежные суффиксы (от длинного к короткому — важен порядок!)
TURKM_PROPER_SUFFIXES = [
    'yndan', 'inden',
    'nyň',  'niň',
    'yň',   'iň',
    'dan',  'den',
    'anda', 'inde',
    'yny',  'ini',
    'yna',  'ine',
    'yda',  'ide',
    'ny',   'ni',
    'na',   'ne',
    'da',   'de',
    'lar',  'ler',
    'sy',   'si',
    'y',    'i',
]

def strip_proper_suffix(word):
    """Убирает туркменский падежный суффикс с конца слова.
    Возвращает (основа, суффикс)."""
    word_n = norm(word)  # нормализуем ň→ñ
    for suf in TURKM_PROPER_SUFFIXES:
        suf_n = norm(suf)
        if word_n.endswith(suf_n) and len(word_n) > len(suf_n) + 2:
            return word[:-len(suf)], suf
    return word, ''

def normalize_proper(word):
    """
    Нормализует имя собственное. Возвращает (нормализованная_форма, суффикс, найдено).
    найдено=True  — слово найдено в PROPER_NORM
    найдено=False — слово с заглавной, основа возвращена как кандидат
    """
    # 1. Прямое совпадение
    if word in PROPER_NORM:
        return PROPER_NORM[word], '', True

    # 2. Убираем суффикс, ищем основу
    base, suf = strip_proper_suffix(word)
    if base in PROPER_NORM:
        return PROPER_NORM[base], suf, True

    # 3. Возвращаем основу как кандидат (для ручного пополнения словаря)
    return base, suf, False


# Тест нормализатора
test_names = ['Kazahyň', 'Russiýanyň', 'Kazahystanda', 'Nazarbaýew', 'Nursultan']
print("🔤 Тест нормализатора имён собственных:")
for name in test_names:
    norm_result, suf, found = normalize_proper(name)
    status = '✅ найден' if found else '⚠️  кандидат'
    suf_str = f" + '{suf}'" if suf else ''
    print(f"  {name:25} → '{norm_result}'{suf_str}  [{status}]")

✅ PROPER_NORM загружен из proper_norm.json: 585 записей
🔤 Тест нормализатора имён собственных:
  Kazahyň                   → 'Kazahystan' + 'yň'  [✅ найден]
  Russiýanyň                → 'Russiýa' + 'nyň'  [✅ найден]
  Kazahystanda              → 'Kazahystan' + 'anda'  [✅ найден]
  Nazarbaýew                → 'Nazarbaýew'  [✅ найден]
  Nursultan                 → 'Nursultan'  [✅ найден]


## 🔬 Ячейка 6 — Основные функции сегментации

In [ ]:
# =============================================================
# ОСНОВНЫЕ ФУНКЦИИ СЕГМЕНТАЦИИ
# =============================================================

# Слова которые НЕ разбиваются (иначе алгоритм даёт ложные разбивки)
NO_SPLIT = {norm(w) for w in [
    'taýýar',   # taý+ýar → неправильно, taýýar = готовый
    'ýaýran',   # ýaý+ran → неправильно
    'aýran',    # аналогично
]}

def find_stem_and_ending(word):
    """
    Ищет стем и окончание.
    Возвращает (сегментированная_строка, тип)
    тип: 'stem_only' | 'stem+ending' | 'proper' | 'proper_candidate' | 'unknown'
    """
    word_norm = norm(word)        # ← нормализация ň→ñ
    word_low  = word_norm.lower()
    word_len  = len(word_norm)
    min_stem  = 2

    # Проверка исключений (слова которые не разбиваются)
    if word_low in NO_SPLIT:
        if word_low in STEMS_SET: return word, '', '', 'stem_only'
        return word, '', '', 'unknown'

    # Шаг 1: стем + окончание (ПРИОРИТЕТ — максимально дробная сегментация)
    # Ищем от самого длинного стема к короткому (min_stem=2)
    if word_len > min_stem:
        for i in range(word_len - min_stem, 0, -1):
            ending_norm = word_norm[word_len - i:]
            stem_norm   = word_norm[:word_len - i]
            if len(stem_norm) >= min_stem and ending_norm in endings_form_set and stem_norm.lower() in STEMS_SET:
                # SUFFIX_INDEX теперь содержит маппинг форма→сегментация
                # Например: 'dyrdy' → 'dyr@@ dy', 'larda' → 'lar@@ da'
                ending_seg = SUFFIX_INDEX.get(ending_norm, ending_norm)
                stem_orig  = word[:word_len - i]
                return stem_orig, ending_norm, ending_seg, 'stem+ending'

    # Шаг 2: слово целиком — стем (только если не разбилось)
    if word_low in STEMS_SET:
        return word, '', '', 'stem_only'

    # Шаг 3: имя собственное — нормализатор
    if word[0].isupper() and len(word) > 1:
        norm_base, suf, found = normalize_proper(word)
        if found:
            if suf:
                suf_norm   = norm(suf)
                ending_seg = SUFFIX_INDEX.get(suf_norm, suf_norm)
                return norm_base, suf, ending_seg, 'proper'
            else:
                return norm_base, '', '', 'proper'
        else:
            return norm_base, suf, suf, 'proper_candidate'

    # Шаг 4: не найдено
    return word, '', '', 'unknown'


def segment_word(word):
    """Сегментирует слово, возвращает строку с @@ разметкой."""
    stem, ending_form, ending_seg, typ = find_stem_and_ending(word)
    if ending_seg:
        return f"{stem}@@ {ending_seg}"
    return stem


def morpho_segment(text):
    """Сегментирует строку текста."""
    tokens = re.findall(r'\w+|[^\w\s]', text)
    result = []
    for token in tokens:
        if norm(token.lower()) in STOP_WORDS:
            result.append(token)
        elif re.match(r'^[a-zA-ZäöüňşçýžäÄÖÜŇŞÇÝŽñÑ]+$', token, re.IGNORECASE):
            result.append(segment_word(token))
        else:
            result.append(token)
    return ' '.join(result)


# Множество нормализованных форм окончаний для быстрого поиска
endings_form_set = set(endings_form)  # уже нормализованы при загрузке

# ── Тест сегментации ────────────────────────────────────────────────────
test_words = [
    ('ýylyň',           'ýyl + yň'),
    ('biziň',           'biz + iň'),
    ('maslahatynyň',    'maslahaty + n@@yň'),
    ('bolup',           'bol + up'),
    ('edildi',          'edil + di'),
    ('diýip',           'diý + ip'),
    ('işleriniň',       'işleri + n@@iň'),
    ('talaplaryny',     'talap + lar@@y@@ny'),
    ('Kazahystanda',    'Kazahystan + da (имя)'),
    ('döwlet',          'стем без окончания'),
]
print("🔬 Тест сегментации:")
ok = sum(1 for w, _ in test_words if find_stem_and_ending(w)[3] != 'unknown')
for w, note in test_words:
    seg = segment_word(w)
    typ = find_stem_and_ending(w)[3]
    mark = '✅' if typ != 'unknown' else '❌'
    print(f"  {mark} {w:25} → {seg:35} [{note}]")
print(f"\n  Покрытие теста: {ok}/{len(test_words)}")


🔬 Тест сегментации:
  ✅ ýylyň                     → ýyl@@ yñ                            [ýyl + yň]
  ✅ biziň                     → biz@@ iñ                            [biz + iň]
  ✅ maslahatynyň              → maslahat@@ yn@@ yñ                  [maslahaty + n@@yň]
  ✅ bolup                     → bol@@ up                            [bol + up]
  ✅ edildi                    → ed@@ il@@ di                        [edil + di]
  ✅ diýip                     → diý@@ ip                            [diý + ip]
  ✅ işleriniň                 → iş@@ ler@@ i@@ niñ                  [işleri + n@@iň]
  ✅ talaplaryny               → talap@@ lar@@ y@@ ny                [talap + lar@@y@@ny]
  ✅ Kazahystanda              → Kazahystan@@ da                     [Kazahystan + da (имя)]
  ✅ döwlet                    → döwlet                              [стем без окончания]

  Покрытие теста: 10/10


In [ ]:
# В Colab перед запуском ячейки 7:
import shutil, os
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)  # удалить старые чанки
os.makedirs(OUTPUT_DIR)

## ▶️ Ячейка 7 — Запуск сегментации по чанкам

In [ ]:
# =============================================================
# ЗАПУСК СЕГМЕНТАЦИИ
# =============================================================

# Читаем входной файл
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    lines = [line.strip() for line in f if line.strip()]

os.makedirs(OUTPUT_DIR, exist_ok=True)
n_chunks = (len(lines) + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"📄 Входной файл:   {INPUT_FILE}")
print(f"📦 Всего строк:    {len(lines)}")
print(f"🗂️  Чанков:         {n_chunks} (по {CHUNK_SIZE} строк)")
print(f"📁 Выходная папка: {OUTPUT_DIR}")
print()

total_time = 0.0

for i in tqdm(range(n_chunks), desc="⏳ Сегментирую"):
    output_path = os.path.join(OUTPUT_DIR, f"chunk_{i:03d}.txt")

    if os.path.exists(output_path):
        print(f"⚡ Чанк {i} уже существует, пропускаем...")
        continue

    chunk_lines = lines[i * CHUNK_SIZE: (i + 1) * CHUNK_SIZE]
    start = time.time()

    segmented = [morpho_segment(line) for line in chunk_lines]

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for line in segmented:
            f_out.write(line + '\n')

    elapsed = time.time() - start
    total_time += elapsed
    print(f"✅ Чанк {i:03d} ({len(chunk_lines)} строк) → {output_path}  [{elapsed:.2f} сек]")

print(f"\n🏁 Готово! Суммарное время: {total_time:.2f} сек")

📄 Входной файл:   turkmen_80000.txt
📦 Всего строк:    80000
🗂️  Чанков:         16 (по 5000 строк)
📁 Выходная папка: turkmen_80000_segmented



⏳ Сегментирую:   6%|▋         | 1/16 [00:00<00:06,  2.28it/s]

✅ Чанк 000 (5000 строк) → turkmen_80000_segmented/chunk_000.txt  [0.44 сек]


⏳ Сегментирую:  12%|█▎        | 2/16 [00:00<00:06,  2.24it/s]

✅ Чанк 001 (5000 строк) → turkmen_80000_segmented/chunk_001.txt  [0.45 сек]


...

✅ Чанк 013 (5000 строк) → turkmen_80000_segmented/chunk_013.txt  [0.77 сек]


⏳ Сегментирую:  94%|█████████▍| 15/16 [00:07<00:00,  1.40it/s]

✅ Чанк 014 (5000 строк) → turkmen_80000_segmented/chunk_014.txt  [0.84 сек]


⏳ Сегментирую: 100%|██████████| 16/16 [00:08<00:00,  1.85it/s]

✅ Чанк 015 (5000 строк) → turkmen_80000_segmented/chunk_015.txt  [0.68 сек]

🏁 Готово! Суммарное время: 8.60 сек


## 🔗 Ячейка 8 — Объединение чанков в один файл

In [ ]:
# =============================================================
# ОБЪЕДИНЯЕМ ВСЕ ЧАНКИ В ОДИН ВЫХОДНОЙ ФАЙЛ
# =============================================================

output_merged = INPUT_FILE.replace('.txt', '_segmented.txt')

chunk_files = sorted([
    os.path.join(OUTPUT_DIR, f)
    for f in os.listdir(OUTPUT_DIR)
    if f.startswith('chunk_') and f.endswith('.txt')
])

total_lines = 0
with open(output_merged, 'w', encoding='utf-8') as out_f:
    for chunk_path in chunk_files:
        with open(chunk_path, 'r', encoding='utf-8') as in_f:
            for line in in_f:
                out_f.write(line)
                total_lines += 1

print(f"✅ Объединено {len(chunk_files)} чанков → {output_merged}")
print(f"   Всего строк: {total_lines}")

✅ Объединено 16 чанков → turkmen_80000_segmented.txt
   Всего строк: 80000


## 📊 Ячейка 9 — Диагностика покрытия

In [ ]:
# ── Перезагружаем словари (на случай обновления файлов) ──────────
print('🔄 Перезагрузка словарей...')
endings_form, endings_seg, SUFFIX_INDEX = load_endings_with_suffix_index(ENDINGS_FILE)
STEMS_SET   = load_stems(STEMS_FILE)
STOP_WORDS  = load_stopwords(STOPWORDS_FILE)
endings_form_set = set(endings_form)
print('✅ Готово\n')

# =============================================================
# ДИАГНОСТИКА: сколько токенов найдены / не найдены
# =============================================================
from collections import Counter

with open(output_merged, 'r', encoding='utf-8') as f:
    seg_lines = f.readlines()

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    orig_lines = [l.strip() for l in f if l.strip()]

stats = Counter()
unknown_words = Counter()

for orig_line, seg_line in zip(orig_lines, seg_lines):
    orig_tokens = re.findall(r'\w+', orig_line)
    for token in orig_tokens:
        if token.lower() in STOP_WORDS:
            stats['stopword'] += 1
            continue
        if not re.match(r'^[a-zA-ZäöüňşçýžäÄÖÜŇŞÇÝŽ]+$', token, re.IGNORECASE):
            stats['non_alpha'] += 1
            continue
        _, _, _, typ = find_stem_and_ending(token)
        stats[typ] += 1
        if typ == 'unknown':
            unknown_words[token] += 1

total = sum(stats.values())
print("📊 Статистика покрытия:")
print(f"  {'Тип':25} {'Кол-во':>8}  {'%':>6}")
print(f"  {'-'*42}")
for typ, cnt in stats.most_common():
    label = {
        'stem_only':       '✅ стем (без окончания)',
        'stem+ending':     '✅ стем + окончание',
        'proper':          '✅ имя собственное',
        'proper_candidate':'⚠️  имя-кандидат',
        'unknown':         '❌ не найдено',
        'stopword':        '🔇 стоп-слово',
        'non_alpha':       '🔢 цифры/знаки',
    }.get(typ, typ)
    print(f"  {label:30} {cnt:>8}   {cnt/total*100:5.1f}%")
print(f"  {'─'*42}")
print(f"  {'ИТОГО':25} {total:>8}  100.0%")

if unknown_words:
    print(f"\n❌ Топ-20 не найденных слов (пополните словарь стемов):")
    for w, c in unknown_words.most_common(20):
        print(f"    '{w}' ({c}x)")

🔄 Перезагрузка словарей...
  ✅ Загружено окончаний: 2416 форм
  ✅ Суффиксных ключей:   2053
  ✅ Загружено стемов: 37620
  ✅ Загружено стоп-слов: 84
✅ Готово

📊 Статистика покрытия:
  Тип                         Кол-во       %
  ------------------------------------------
  ✅ стем + окончание               573144    46.3%
  ✅ стем (без окончания)           385114    31.1%
  🔇 стоп-слово                     115192     9.3%
  ❌ не найдено                      63451     5.1%
  🔢 цифры/знаки                     55409     4.5%
  ⚠️  имя-кандидат                  28272     2.3%
  ✅ имя собственное                 17765     1.4%
  ──────────────────────────────────────────
  ИТОГО                      1238347  100.0%

❌ Топ-20 не найденных слов (пополните словарь стемов):
    'unutmaly' (30x)
    'razylaşdylar' (30x)
    'gepleşip' (30x)
    'goldaýandygyny' (30x)
    'transfer' (30x)
    'makul' (30x)
    'gazygyň' (30x)
    'döretjekdir' (30x)
    'döredilýändigini' (30x)
    'resmileşdirilme

## ➕ Ячейка 10 — Пополнение словаря имён собственных (интерактивно)

In [ ]:
# =============================================================
# ПОПОЛНЕНИЕ PROPER_NORM НОВЫМИ ИМЕНАМИ
# Показывает ТОЛЬКО настоящие имена собственные —
# обычные слова с заглавной (Biz, Bu, Şeýle...) отфильтрованы.
# =============================================================

from collections import Counter
proper_candidates = Counter()
for orig_line in orig_lines:
    for token in re.findall(r'\w+', orig_line):
        if not (token[0].isupper() and len(token) > 1):
            continue
        if re.match(r'^\d', token):
            continue
        # Пропускаем если слово (строчная форма) уже есть в словаре стемов
        if norm(token.lower()) in STEMS_SET:
            continue
        # Пропускаем если уже в PROPER_NORM
        _, _, found = normalize_proper(token)
        if found:
            continue
        # Реальный кандидат — имя которого нет ни в стемах ни в PROPER_NORM
        base, _ = strip_proper_suffix(token)
        # Если основа (без суффикса) тоже есть в стемах — пропускаем
        if norm(base.lower()) in STEMS_SET:
            continue
        proper_candidates[base] += 1

if proper_candidates:
    print('⚠️  Имена-кандидаты для добавления в PROPER_NORM:')
    print('    (скопируйте нужные в словарь PROPER_NORM в ячейке 5,')
    print('     затем перезапустите ячейки 5→6→7→8→9)')
    print()
    for name, cnt in proper_candidates.most_common(30):
        print(f"    '{name}': '{name}',   # {cnt}x")
else:
    print('✅ Новых имён-кандидатов не найдено!')


⚠️  Имена-кандидаты для добавления в PROPER_NORM:
    (скопируйте нужные в словарь PROPER_NORM в ячейке 5,
     затем перезапустите ячейки 5→6→7→8→9)

    'Maliýa': 'Maliýa',   # 48x
    'Karim': 'Karim',   # 36x
    'Atyrau': 'Atyrau',   # 25x
    'Jemaat': 'Jemaat',   # 19x
    'İşler': 'İşler',   # 19x
    'Мәңгілік': 'Мәңгілік',   # 17x
    'Ватан': 'Ватан',   # 17x
    'Ukrainanıñ': 'Ukrainanıñ',   # 16x
    'Konstitusiýas': 'Konstitusiýas',   # 15x
    'Abdulla': 'Abdulla',   # 14x
    'Medeu': 'Medeu',   # 14x
    'Nusypov': 'Nusypov',   # 14x
    'Ýatgyt': 'Ýatgyt',   # 14x
    'Asambleýa': 'Asambleýa',   # 14x
    'Görüşüşime': 'Görüşüşime',   # 14x
    'Kürgizstan': 'Kürgizstan',   # 14x
    'Isikeshew': 'Isikeshew',   # 14x
    'Wezig': 'Wezig',   # 14x
    'RF': 'RF',   # 14x
    'Öýler': 'Öýler',   # 14x
    'Fund': 'Fund',   # 14x
    'Awganistan': 'Awganistan',   # 14x
    'Kaşkaşstan': 'Kaşkaşstan',   # 14x
    'Tulpar': 'Tulpar',   # 14x
    'Skills': 'Skills',   # 14x